# Day 11: Context Window Management, Walked Through

An LLM's context window is a hard limit on how much text fits in one call --
system prompt, retrieved documents, the question, and room for the answer all have
to share it. This notebook walks through budgeting, selection strategies,
compression, token counting, iterative optimization, multi-turn handling, and
proof that more context isn't automatically better.

Run each cell in order. Plain Python throughout, self-contained.

## Cell 1: Why context windows matter

Retrieval can find dozens of relevant chunks, but only so much text fits in one
LLM call. Past that limit, content gets rejected or truncated -- and even when
everything technically fits, cramming in marginal material can make the answer
worse, not better, as we'll measure directly later in this notebook.

## Cell 2: Calculate context budgets

Work out exactly how much room is left for documents once everything else is
accounted for.

In [ ]:
def estimate_tokens(text):
    return int(len(text.split()) * 1.3)

def context_budget(model_window, system_prompt, query, reserved_response_tokens):
    system_tokens = estimate_tokens(system_prompt)
    query_tokens = estimate_tokens(query)
    fixed_cost = system_tokens + query_tokens + reserved_response_tokens
    return max(0, model_window - fixed_cost), fixed_cost

system_prompt = "You are a helpful assistant that answers using ONLY the provided context."
query = "What is RAG and why does it matter?"

for window in [4096, 128_000]:
    remaining, fixed = context_budget(window, system_prompt, query, reserved_response_tokens=500)
    print(f"{window:>7,}-token window: {fixed} tokens fixed cost, {remaining:,} tokens left for documents")

## Cell 3: Document selection strategies

When not everything fits, HOW you choose matters. Relevance-first can waste
budget on near-duplicates; coverage-first guarantees breadth.

In [ ]:
candidates = [
    {"title": "RAG overview A", "text": "RAG retrieves documents before generating an answer. " * 20, "similarity": 0.95, "topic": "rag_basics"},
    {"title": "RAG overview B", "text": "RAG combines retrieval with generation for grounded answers. " * 20, "similarity": 0.93, "topic": "rag_basics"},
    {"title": "Vector DB", "text": "Vector databases store embeddings for fast search. " * 20, "similarity": 0.70, "topic": "vector_db"},
    {"title": "Chunking", "text": "Chunking splits documents into searchable pieces. " * 20, "similarity": 0.60, "topic": "chunking"},
]

def fit_to_budget(ordered, budget):
    selected, used = [], 0
    for c in ordered:
        cost = estimate_tokens(c["text"])
        if used + cost <= budget:
            selected.append(c)
            used += cost
    return selected, used

budget = 400
by_relevance, _ = fit_to_budget(sorted(candidates, key=lambda c: c["similarity"], reverse=True), budget)

topics_seen = set()
by_coverage_order = []
for c in sorted(candidates, key=lambda c: c["similarity"], reverse=True):
    if c["topic"] not in topics_seen:
        by_coverage_order.append(c)
        topics_seen.add(c["topic"])
by_coverage, _ = fit_to_budget(by_coverage_order, budget)

print("By relevance:", [c["title"] for c in by_relevance], "- topics:", {c["topic"] for c in by_relevance})
print("By coverage: ", [c["title"] for c in by_coverage], "- topics:", {c["topic"] for c in by_coverage})

## Cell 4: Summarization techniques

Extractive summarization pulls real sentences out -- a strict "never invent
anything" guarantee, at the cost of some compression power.

In [ ]:
import re

def extractive_summarize(text, max_sentences=2):
    raw_sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]
    seen, sentences = set(), []
    for s in raw_sentences:
        if s not in seen:
            seen.add(s)
            sentences.append(s)
    if len(sentences) <= max_sentences:
        return " ".join(sentences)
    words = re.findall(r'\w+', text.lower())
    freq = {}
    for w in words:
        freq[w] = freq.get(w, 0) + 1
    def score(s):
        ws = re.findall(r'\w+', s.lower())
        return sum(freq.get(w, 0) for w in ws) / max(1, len(ws))
    scored = [(i, s, score(s)) for i, s in enumerate(sentences)]
    top = sorted(scored, key=lambda item: item[2], reverse=True)[:max_sentences]
    return " ".join(s for _, s, _ in sorted(top, key=lambda item: item[0]))

document = (
    "RAG stands for Retrieval-Augmented Generation. It retrieves relevant documents "
    "before generating an answer. This matters because language models have a "
    "knowledge cutoff date. RAG lets them use fresh information without retraining."
)

summary = extractive_summarize(document, max_sentences=2)
print(f"Original: {estimate_tokens(document)} tokens")
print(f"Summary:  {estimate_tokens(summary)} tokens ({estimate_tokens(summary)/estimate_tokens(document):.0%} of original)")
print(f"\n\"{summary}\"")

## Cell 5: Token counting

A simple word-based estimate vs. a real tokenizer (tiktoken, if installed).

In [ ]:
try:
    import tiktoken
    encoding = tiktoken.get_encoding("cl100k_base")
    TIKTOKEN_AVAILABLE = True
except ImportError:
    TIKTOKEN_AVAILABLE = False

samples = {
    "short": "What is RAG?",
    "prose": "RAG retrieves relevant documents before generating an answer.",
    "punctuation_heavy": "Pricing: $0.00002/1K tokens, 128,000 token limit, 99.9% uptime!!!",
}

for name, text in samples.items():
    word_est = estimate_tokens(text)
    if TIKTOKEN_AVAILABLE:
        real = len(encoding.encode(text))
        print(f"{name:<20}: word-estimate={word_est:>3}, real={real:>3}, error={((word_est-real)/real*100):+.0f}%")
    else:
        print(f"{name:<20}: word-estimate={word_est:>3} (tiktoken not installed -- install it to see real counts)")

## Cell 6: Iterative optimization

Start with the best documents; if one doesn't fit, try summarizing it before
giving up on it entirely.

In [ ]:
docs = [
    {"title": "RAG overview", "text": "RAG retrieves documents before generating an answer, grounding responses in real sources instead of pure memorized knowledge.", "similarity": 0.95},
    {"title": "Why RAG matters", "text": "RAG matters because language models can be outdated. Retrieval lets them use fresh, specific information without needing to be retrained.", "similarity": 0.85},
    {"title": "Vector databases", "text": "A vector database stores embeddings and allows fast similarity search across large document collections, often using approximate indexing.", "similarity": 0.70},
]

def iterative_fit(candidates, budget):
    ranked = sorted(candidates, key=lambda c: c["similarity"], reverse=True)
    selected, used = [], 0
    for c in ranked:
        full_cost = estimate_tokens(c["text"])
        remaining = budget - used
        if full_cost <= remaining:
            selected.append((c["title"], "full"))
            used += full_cost
        else:
            summary = extractive_summarize(c["text"], max_sentences=1)
            summary_cost = estimate_tokens(summary)
            if summary_cost <= remaining:
                selected.append((c["title"], "summarized"))
                used += summary_cost
            else:
                selected.append((c["title"], "dropped"))
    return selected, used

for budget in [40, 80]:
    selected, used = iterative_fit(docs, budget)
    print(f"Budget {budget}: used {used} tokens")
    for title, status in selected:
        print(f"  {title}: {status}")

## Cell 7: Multi-turn strategy

When one call genuinely can't fit everything, carry a bounded SUMMARY forward
between turns instead of the full text of every prior turn.

In [ ]:
turn_docs = [
    {"title": "RAG overview", "text": "RAG retrieves documents before generating an answer. It grounds responses in real sources."},
    {"title": "Vector databases", "text": "A vector database stores embeddings for fast similarity search. It finds documents by meaning."},
    {"title": "Chunking", "text": "Chunking splits documents into smaller pieces. This helps them fit well in embeddings."},
]

running_summary = ""
for i, doc in enumerate(turn_docs, start=1):
    combined = running_summary + " " + doc["text"]
    running_summary = extractive_summarize(combined, max_sentences=2)
    print(f"Turn {i} ({doc['title']}): running summary -> \"{running_summary}\"")

Worth noticing: the running summary can stay nearly identical across turns, because
the extractive summarizer keeps re-picking high-scoring sentences from EARLIER
turns rather than reliably folding in each turn's new content. That's a real
limitation of naive extractive summarization as a "carry context forward"
mechanism -- a genuinely abstractive summarizer (a real LLM call) handles this much
more reliably, at the cost of needing an actual API call to do it.

## Cell 8: Quality analysis

Proof that more context isn't automatically better: topical relevance density
PEAKS early and then declines as off-topic documents get added.

In [ ]:
STOPWORDS = {"what", "is", "a", "an", "the", "and", "for", "how", "do", "does", "why"}

def keywords(text):
    return {w.strip("?.,!").lower() for w in text.split()} - STOPWORDS

query = "What is RAG?"
documents_by_relevance = [
    {"title": "What is RAG", "text": "RAG stands for Retrieval-Augmented Generation. It retrieves documents before generating an answer."},
    {"title": "Why RAG Matters", "text": "RAG matters because language models can be outdated."},
    {"title": "Vector Databases", "text": "A vector database stores embeddings and allows fast similarity search."},
    {"title": "Python Data Types", "text": "Python has several built-in data types including strings and integers."},
]

query_words = keywords(query)
for count in range(1, len(documents_by_relevance) + 1):
    included = documents_by_relevance[:count]
    total_words = sum(len(d["text"].split()) for d in included)
    on_topic_words = sum(len(d["text"].split()) for d in included if query_words & keywords(d["text"]))
    density = on_topic_words / total_words
    print(f"{count} doc(s): topical density = {density:.2f}  ({[d['title'] for d in included]})")

Notice density peaks with just the first document and steadily drops as unrelated
documents get concatenated in -- more context, worse on-topic focus, at a higher
token cost.

## Cell 9: Key takeaways

- A context window is a hard, shared budget: system prompt, documents, query, and
  response space all have to fit together.
- Selection strategy matters: pure relevance-first can waste budget on
  near-duplicate chunks; coverage-first guarantees breadth across topics.
- Extractive summarization compresses with a strict no-fabrication guarantee;
  abstractive summarization can compress further but needs its grounding checked.
- Word-based token estimates can be off by 20-50%+ depending on content type --
  use a real tokenizer once token budgets genuinely matter.
- Optimization is iterative: try full, fall back to summarized, only drop entirely
  when neither fits.
- Multi-turn strategies can bypass a hard context limit by carrying forward a
  bounded summary instead of the full history -- at the cost of more calls and
  more complexity.
- More context is NOT automatically better: topical relevance density can peak
  early and decline as less-relevant material gets added, even while a naive
  relevance check stays blind to the problem.
- Spend context deliberately -- every token given to a marginal document is a
  token not spent on response quality, and can actively dilute the answer.